# Data cleaning of the rental listings

**Input:** `data/data_processed/flatfox_rentals_zentralschweiz.csv` (from `data_engineering_real_estate.ipynb`: merged scraping runs, municipality assignment, municipal indicators)

**Output:**
- `data/data_processed/flatfox_rentals_clean.csv` – cleaned analysis sample
- `data/data_processed/flatfox_rentals_excluded.csv` – every removed listing with its reason (for documentation)

**Target variable:** monthly **net rent** (as defined in the feasibility study) and the net rent per m² derived from it.

The notebook follows the six mandatory steps of the project description:

1. Missing values and duplicates
2. Data types
3. Value ranges (plausibility)
4. Outliers
5. Enrichment with additional columns
6. Format for the analysis

Beforehand, the sample is restricted to unfurnished, non-temporary rental apartments with a monthly price, because the research questions concern apartments. The input contains all listings in Central Switzerland (including sales, commercial units and parking spaces). All filters based on listing characteristics are applied in this notebook.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "data_processed"
INPUT_FILE = DATA_DIR / "flatfox_rentals_zentralschweiz.csv"
OUTPUT_FILE = DATA_DIR / "flatfox_rentals_clean.csv"
EXCLUDED_FILE = DATA_DIR / "flatfox_rentals_excluded.csv"
PLZ_FILE = PROJECT_DIR / "data" / "geodata" / "ortschaftenverzeichnis_plz.csv"

CENTRAL_CANTONS = ["LU", "UR", "SZ", "OW", "NW", "ZG"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

raw = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded: {len(raw)} listings, {raw.shape[1]} columns")

In [ ]:
excluded = []
log = [("Initial dataset", len(raw))]


def exclude(df, mask, step, reason):
    # Removes the rows selected by the mask and logs them.
    mask = mask.fillna(False)
    removed = df.loc[mask, ["pk", "gemeinde_name", "kanton_geo", "object_type", "surface_living", "number_of_rooms"]].copy()
    removed["step"] = step
    removed["reason"] = reason
    excluded.append(removed)
    kept = df.loc[~mask].copy()
    log.append((f"{step}: {reason}", len(kept)))
    print(f"{step} – {reason}: {mask.sum()} removed, {len(kept)} remaining")
    return kept

## Restriction to rental apartments

The input contains all listings in Central Switzerland. Only rental apartments that are comparable with the regular rental market are relevant for the research questions:

| Criterion | Removed | Reason |
|---|---|---|
| Offer type | sales | purchase prices are not comparable with rents |
| Object type | commercial units, parking spaces, shared rooms, houses | not comparable with apartments |
| Furnishing | furnished apartments (`is_furnished` or object type `FURNISHED_FLAT`) | the rent includes the furniture |
| Duration | temporary offers (`is_temporary`) | often sublets at special prices |
| Price unit | everything except a monthly price | e.g. annual price per m² for commercial space |
| Coordinates | listings whose postal code belongs to a different canton than the coordinates | municipality and municipal indicators would be wrong |

**Coordinate check:** The municipality is determined from the coordinates in `data_engineering_real_estate.ipynb`. Some listings have coordinates that do not match the stated address (e.g. address in Pratteln BL, coordinates in Lucerne). We therefore compare the canton of the coordinates with the cantons of the postal code according to the official swisstopo postal code directory (`data/geodata/ortschaftenverzeichnis_plz.csv`). The comparison is done at canton level rather than municipality level, because postal code areas often extend across municipal boundaries. Listings with a postal code that is missing from the directory (e.g. 6000 Luzern) are not removed.

In [ ]:
display(raw["object_type"].value_counts().rename("count").to_frame())

APARTMENT_TYPES = ["APARTMENT", "ATTIC_FLAT", "DUPLEX", "ROOF_FLAT", "STUDIO", "LOFT", "TERRACE_FLAT"]

# Cantons per postal code; a postal code can belong to several cantons.
plz = pd.read_csv(PLZ_FILE, sep=";", encoding="utf-8-sig")
cantons_by_plz = plz.groupby("PLZ4")["Kantonskürzel"].agg(set)
listing_plz_cantons = pd.to_numeric(raw["zipcode"], errors="coerce").map(cantons_by_plz)
plz_mismatch = pd.Series(
    [isinstance(cantons, set) and canton not in cantons
     for canton, cantons in zip(raw["kanton_geo"], listing_plz_cantons)],
    index=raw.index,
)

df = exclude(raw, raw["offer_type"].ne("RENT"), "0 Scope", "not a rental offer (sale)")
df = exclude(df, ~df["object_type"].isin(APARTMENT_TYPES + ["FURNISHED_FLAT"]), "0 Scope", "not an apartment (commercial, parking, shared room, house)")
df = exclude(df, df["is_furnished"].eq(True) | df["object_type"].eq("FURNISHED_FLAT"), "0 Scope", "furnished")
df = exclude(df, df["is_temporary"].eq(True), "0 Scope", "temporary")
df = exclude(df, df["price_unit"].ne("monthly"), "0 Scope", "no monthly price")

display(df.loc[plz_mismatch.loc[df.index], ["pk", "street", "zipcode", "city", "gemeinde_name", "kanton_geo"]])
df = exclude(df, plz_mismatch.loc[df.index], "0 Scope", "postal code does not match the canton of the coordinates")

apartments = df.copy()

## Missing values and duplicates

### Net rent
The net rent is not stated directly in every listing. Where it is missing but the gross rent and the service charges are available, it is calculated: **net rent = gross rent − service charges**. Listings for which no net rent can be determined are removed. Missing values are not imputed, because the rent is our target variable.

### Living area and number of rooms
Without the living area the rent per m² cannot be calculated, and without the number of rooms a control variable of the analysis is missing. These listings are removed as well and not imputed.

In [ ]:
key_columns = ["rent_net", "rent_gross", "rent_charges", "surface_living", "number_of_rooms",
               "floor", "year_built", "year_renovated", "price_display"]
missing = pd.DataFrame({
    "missing": df[key_columns].isna().sum(),
    "missing (%)": df[key_columns].isna().mean().mul(100).round(1),
})
missing

In [ ]:
df["rent_net_source"] = np.where(df["rent_net"].notna(), "reported", None)
derived = df["rent_net"].isna() & df["rent_gross"].notna() & df["rent_charges"].notna()
df.loc[derived, "rent_net"] = df.loc[derived, "rent_gross"] - df.loc[derived, "rent_charges"]
df.loc[derived, "rent_net_source"] = "derived (gross − charges)"

print(df["rent_net_source"].value_counts(dropna=False))

df = exclude(df, df["rent_net"].isna(), "1 Missing values", "net rent cannot be determined")
df = exclude(df, df["surface_living"].isna(), "1 Missing values", "no living area")
df = exclude(df, df["number_of_rooms"].isna(), "1 Missing values", "no number of rooms")

How much the sample shrinks per canton because of the net rent matters for the interpretation (especially for Uri, Obwalden and Nidwalden):

In [ ]:
before = apartments["kanton_geo"].value_counts()
after = df["kanton_geo"].value_counts()
pd.DataFrame({"apartments": before, "with net rent": after}).reindex(CENTRAL_CANTONS).fillna(0).astype(int)

### Municipal indicators
The social assistance rate is missing for some small municipalities. These listings are not removed, because all other information is available. They drop out automatically in analyses that use the social assistance rate.

In [ ]:
indicator_columns = ["income_median_chf", "population_growth_pct", "population_density_productive",
                     "household_size_mean", "vacancy_rate_pct", "social_assistance_rate_pct",
                     "population_age65plus_pct", "employment_per_100_residents",
                     "new_dwellings_per_1000_residents", "single_family_buildings_pct",
                     "foreign_population_pct", "net_migration_per_1000_residents"]
df[indicator_columns].isna().sum().rename("missing values").to_frame()

### Duplicates
The listing ID (`pk`) is already unique after the merge. However, the same apartment can be listed again under a new ID. A listing counts as a duplicate if

- municipality, street, living area, number of rooms and floor match another listing and
- the other listing only appears after this one has disappeared.

The newest listing is kept, because it contains the current price. The rent is deliberately not part of the comparison, so that relistings with a changed price are also detected.

If two listings were online at the same time, they are different apartments (e.g. identical floor plans in the same new building) and both are kept. Listings without a street or with the placeholder "Auf Anfrage" (on request) are not compared, because municipality, area and number of rooms alone do not identify an apartment.

In [ ]:
assert df["pk"].is_unique, "pk is not unique"

dup_key = ["gemeinde_bfs", "street", "surface_living", "number_of_rooms", "floor"]
df = df.sort_values(["first_seen_at", "pk"], kind="stable")

# "Auf Anfrage" is a placeholder and does not identify an apartment.
street_known = df["street"].notna() & df["street"].ne("Auf Anfrage")
known = df.loc[street_known]

# Duplicate: the next occurrence of the same apartment only appears after this listing has disappeared.
next_first_seen = known.groupby(dup_key, dropna=False)["first_seen_at"].shift(-1)
is_dup = next_first_seen.gt(known["last_seen_at"]).reindex(df.index, fill_value=False)

display(known[known.duplicated(subset=dup_key, keep=False)]
        .sort_values(dup_key + ["first_seen_at"])
        [["pk", "gemeinde_name", "street", "surface_living", "number_of_rooms", "floor", "rent_net", "first_seen_at", "last_seen_at"]]
        .assign(removed=lambda t: t.index.isin(is_dup[is_dup].index)))

df = exclude(df, is_dup, "1 Duplicates", "same apartment under a new ID")

## Data types

When reading from CSV, pandas does not detect all types correctly (e.g. timestamps as text, IDs as floats). We set them explicitly.

In [ ]:
print("Before:")
check_columns = ["pk", "gemeinde_bfs", "number_of_rooms", "floor", "year_built", "first_seen_at",
                 "published", "is_furnished", "attribute_reported_lift", "kanton_geo", "object_type"]
display(df[check_columns].dtypes.rename("dtype").to_frame())

df["pk"] = df["pk"].astype("int64")
df["gemeinde_bfs"] = df["gemeinde_bfs"].astype("Int64")

for col in ["floor", "year_built", "year_renovated"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in ["rent_net", "rent_gross", "rent_charges", "surface_living", "number_of_rooms"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

for col in ["first_seen_at", "last_seen_at"]:
    df[col] = pd.to_datetime(df[col], utc=True)
df["published"] = pd.to_datetime(df["published"], utc=True, format="ISO8601")

bool_columns = ["is_furnished", "is_temporary"] + [c for c in df.columns if c.startswith("attribute_reported_")]
for col in bool_columns:
    df[col] = df[col].astype("boolean")

for col in ["kanton_geo", "object_type", "rent_net_source"]:
    df[col] = df[col].astype("category")

print("After:")
display(df[check_columns].dtypes.rename("dtype").to_frame())

## Value ranges

We check whether the values lie in a plausible range. The limits are deliberately generous; they are only meant to catch obvious data entry errors. Unusual but possible values are handled in the next step (outliers).

| Variable | Plausible range | Reason | Treatment |
|---|---|---|---|
| Living area | ≥ 20 m² | smaller units are rooms, not apartments | remove listing |
| Area per room | ≥ 8 m² | smaller values indicate an input error | remove listing |
| Net rent | > 0 and > service charges | service charges higher than the rent are implausible | remove listing |
| Number of rooms | 1–10 | | remove listing |
| Floor | −1 to 30 | the tallest residential buildings in the region have < 30 floors; 99 is a placeholder | set value to missing |
| Year built | 1500 to 2027 | from old-town buildings to new buildings under construction | set value to missing |
| Coordinates | within Central Switzerland | already checked during the municipality assignment | – |

For floor and year built we only set the value to missing, because these variables are not central and the rest of the listing is usable.

In [ ]:
display(df[["surface_living", "number_of_rooms", "rent_net", "rent_charges", "floor", "year_built"]]
        .describe().T[["min", "25%", "50%", "75%", "max"]])

df = exclude(df, df["surface_living"] < 20, "3 Value ranges", "living area < 20 m²")
df = exclude(df, df["surface_living"] / df["number_of_rooms"] < 8, "3 Value ranges", "area per room < 8 m²")
df = exclude(df, (df["rent_net"] <= 0) | (df["rent_charges"] >= df["rent_net"]), "3 Value ranges", "net rent ≤ 0 or ≤ service charges")
df = exclude(df, ~df["number_of_rooms"].between(1, 10), "3 Value ranges", "number of rooms outside 1–10")

bad_floor = ~df["floor"].between(-1, 30) & df["floor"].notna()
bad_year = ~df["year_built"].between(1500, 2027) & df["year_built"].notna()
print(f"Floor set to missing: {bad_floor.sum()} (values: {sorted(df.loc[bad_floor, 'floor'].tolist())})")
print(f"Year built set to missing: {bad_year.sum()}")
df.loc[bad_floor, "floor"] = pd.NA
df.loc[bad_year, "year_built"] = pd.NA

## Outliers

For the outliers we look at the **net rent per m²**, because it makes apartments of different sizes comparable.

**Method:** Tukey's rule (values below Q1 − 1.5·IQR or above Q3 + 1.5·IQR), applied to the **logarithm** of the rent per m². Rents are right-skewed; without the log transformation too many expensive but genuine apartments would count as outliers.

**Treatment:** Remove. Only a few listings are affected (see table). At the lower end there are very large areas at a low rent, which points to an incorrectly recorded area (e.g. including ancillary rooms). At the upper end there are luxury or very small apartments that do not belong to the regular rental market. Such extreme values would have a disproportionate influence on the regression.

In [ ]:
df["rent_net_per_m2"] = df["rent_net"] / df["surface_living"]
log_price = np.log(df["rent_net_per_m2"])

q1, q3 = log_price.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = np.exp(q1 - 1.5 * iqr), np.exp(q3 + 1.5 * iqr)
is_outlier = ~df["rent_net_per_m2"].between(lower, upper)

print(f"Bounds: {lower:.1f} to {upper:.1f} CHF/m² per month")
display(df.loc[is_outlier, ["pk", "gemeinde_name", "object_type", "number_of_rooms", "surface_living",
                            "rent_net", "rent_net_per_m2", "short_title"]].sort_values("rent_net_per_m2"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["rent_net_per_m2"], bins=40, color="#3a6ea5", alpha=0.8)
for x in (lower, upper):
    axes[0].axvline(x, color="#c0392b", ls="--")
axes[0].set(title="Net rent per m² with outlier bounds", xlabel="Net rent (CHF/m² per month)", ylabel="Listings")

axes[1].boxplot([df["rent_net_per_m2"], df.loc[~is_outlier, "rent_net_per_m2"]], orientation="horizontal")
axes[1].set_yticks([1, 2], ["before cleaning", "after cleaning"])
axes[1].set(title="Distribution before and after removing the outliers", xlabel="Net rent (CHF/m² per month)")

plt.tight_layout()
plt.show()

df = exclude(df, is_outlier, "4 Outliers", f"net rent/m² outside {lower:.1f}–{upper:.1f} CHF")

*Note on the box plot:* The box plot uses the Tukey bounds on the normal scale and therefore still shows individual points above about 43 CHF/m² after cleaning. On the log scale, which we use for the outlier rule, these values lie within the bounds. They are expensive but plausible apartments (mainly in Zug and the city of Lucerne).

## Enrichment

Additional columns that are useful for the analysis:

| New column | Content |
|---|---|
| `rent_net_per_m2` | net rent per m² and month (already created in step 4) – main target variable |
| `rent_net_source` | whether the net rent was reported or derived (step 1) |
| `charges_share_pct` | share of service charges in the gross rent |
| `building_age` | years since construction or last renovation |
| `room_class` | room category (1–1.5, 2–2.5, 3–3.5, 4–4.5, 5+) for group comparisons |
| `population_density_log` | log of the population density (the density is highly skewed) |
| `days_online` | days between publication and last sighting (indication of demand) |

In [ ]:
rent_gross_full = df["rent_gross"].fillna(df["rent_net"] + df["rent_charges"])
df["charges_share_pct"] = (df["rent_charges"] / rent_gross_full * 100).round(1)

last_year = df[["year_built", "year_renovated"]].max(axis=1)
df["building_age"] = (2026 - last_year).clip(lower=0).astype("Int64")

df["room_class"] = pd.cut(df["number_of_rooms"], bins=[0, 1.5, 2.5, 3.5, 4.5, 11],
                          labels=["1–1.5", "2–2.5", "3–3.5", "4–4.5", "5+"])

df["population_density_log"] = np.log(df["population_density_productive"])

df["days_online"] = (df["last_seen_at"] - df["published"]).dt.days

df[["rent_net_per_m2", "charges_share_pct", "building_age", "room_class", "population_density_log", "days_online"]].describe(include="all").T

## Format for the analysis

The input has more than 150 columns (all fields of the Flatfox API). For the analysis we only keep the required columns in a meaningful order: identification > rent > apartment characteristics > location > municipal indicators > data collection. The merge with the municipal indicators was already done in `data_engineering_real_estate.ipynb`.

In [ ]:
FINAL_COLUMNS = [
    # Identification
    "pk", "source_url", "object_type",
    # Rent
    "rent_net", "rent_net_source", "rent_net_per_m2", "rent_charges", "rent_gross", "charges_share_pct",
    # Apartment characteristics
    "surface_living", "number_of_rooms", "room_class", "floor", "year_built", "year_renovated", "building_age",
    "attribute_reported_balconygarden", "attribute_reported_lift", "attribute_reported_view",
    "attribute_reported_parkingspace", "attribute_reported_minergie",
    # Location
    "latitude", "longitude", "gemeinde_bfs", "gemeinde_name", "kanton_geo",
    # Municipal indicators
    *indicator_columns, "population_density_log",
    # Data collection
    "published", "first_seen_at", "last_seen_at", "scrape_count", "days_online",
]

clean = (df[FINAL_COLUMNS]
         .sort_values(["kanton_geo", "gemeinde_name", "pk"])
         .reset_index(drop=True))

print(f"Clean dataset: {len(clean)} listings, {clean.shape[1]} columns, "
      f"{clean['gemeinde_bfs'].nunique()} municipalities")
clean.head()

## Result and saving

In [ ]:
summary = pd.DataFrame(log, columns=["step", "remaining listings"])
summary["removed"] = (-summary["remaining listings"].diff()).fillna(0).astype(int)
display(summary)

print("Listings and municipalities per canton:")
display(clean.groupby("kanton_geo", observed=True)
        .agg(listings=("pk", "size"), municipalities=("gemeinde_bfs", "nunique"))
        .reindex(CENTRAL_CANTONS))

## Validation

Before saving, a few assertions check the final dataset and the exclusion log: every listing of the input ends up either in the clean dataset or, with exactly one reason, in the excluded listings. The expected row counts are only a snapshot of the current data and trigger a warning, not an error.

In [ ]:
excluded_df = pd.concat(excluded, ignore_index=True)

# Snapshot of the data collected from 18 October to 6 November 2026, not a fixed requirement.
EXPECTED_ROWS = {"input": 2640, "clean": 510}

assert clean["pk"].is_unique, "pk is not unique in the clean dataset"
assert clean[["rent_net", "surface_living", "number_of_rooms"]].notna().all().all(), "Missing rent, area or rooms"
assert np.isfinite(clean["rent_net_per_m2"]).all() and clean["rent_net_per_m2"].gt(0).all(), "rent_net_per_m2 not finite and positive"

assert len(clean) + len(excluded_df) == len(raw), "Clean + excluded listings do not add up to the input"
assert excluded_df["pk"].is_unique, "A listing was excluded more than once"
assert excluded_df[["step", "reason"]].notna().all().all(), "Excluded listing without reason"
assert set(clean["pk"]).isdisjoint(excluded_df["pk"]), "Listing both clean and excluded"
assert set(clean["pk"]) | set(excluded_df["pk"]) == set(raw["pk"]), "Listing lost during cleaning"

for name, actual in {"input": len(raw), "clean": len(clean)}.items():
    if actual != EXPECTED_ROWS[name]:
        warnings.warn(f"{name}: {actual} rows instead of {EXPECTED_ROWS[name]} (snapshot). "
                      "Update the snapshot and the interpretation texts if the data changed.")

print(f"Validation passed: {len(raw)} input = {len(clean)} clean + {len(excluded_df)} excluded "
      f"({excluded_df['reason'].nunique()} different reasons)")

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
excluded_df.to_csv(EXCLUDED_FILE, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_FILE.relative_to(PROJECT_DIR)} ({len(clean)} rows)")
print(f"Saved: {EXCLUDED_FILE.relative_to(PROJECT_DIR)} ({len(excluded_df)} rows)")